# Dependency Injection Basics

This notebook covers:

1. Inject a dependency into a route with `Depends(...)`
2. Chain dependencies (a dep that consumes another dep)
3. Use class-callable dependencies to bundle related parameters
4. Understand per-request caching: a dep is computed once per request
5. Clean up resources with `yield` dependencies

**Scope**: FastAPI + `TestClient` against the in-process app. No external services.

DI in FastAPI is *not* a heavy framework — it's a small, predictable pattern: any callable can be a dependency, and `Depends(...)` is how a route asks for one. By the end of this notebook you'll see that the same building block solves "give me the current settings", "give me a paginated query", and "give me an open database connection that's automatically closed after the request".

## 1. The `Depends(...)` Pattern

A FastAPI route is a function. Some of its parameters come from the request (path, query, body, header). Others come from **dependencies**: side-functions FastAPI calls before invoking the route, whose return value gets passed in as an argument.

```python
@app.get("/items")
def list_items(repo: Repo = Depends(get_repo)):
    return repo.all()
```

Mental model: `Depends(get_repo)` says "before running `list_items`, run `get_repo()`; pass its return value as `repo`." The route is still a plain function — DI is just a way to say "I need this thing, please prepare it for me."

What you get for free:

- **Reuse**: any number of routes can `Depends(get_repo)` without redefining how a repo is built.
- **Per-request lifecycle**: the dep runs once per request, regardless of how many places in the route reference it.
- **Testability**: in tests, `app.dependency_overrides[get_repo] = ...` swaps the implementation without touching the route or the request. We use this throughout chapter 7.
- **OpenAPI integration**: parameters declared in a dep show up in the route's OpenAPI schema.

## 2. A Simple Dependency

The smallest useful dep: a function with no parameters returning something the route wants. We'll model the portfolio settings — currency, default page size — that several routes will want to read.

In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app = FastAPI()

# A dep is just a callable. Annotated return type is for IDE / readability;
# FastAPI uses it for OpenAPI but doesn't require it.
def get_settings() -> dict:
    return {"currency": "USD", "default_page_size": 20}

@app.get("/config")
def read_config(settings: dict = Depends(get_settings)):
    return settings

client = TestClient(app)
r = client.get("/config")
print(r.status_code, r.json())

Three things just happened:

1. `Depends(get_settings)` told FastAPI: "before calling `read_config`, call `get_settings()` and pass its return value as `settings`."
2. The route's signature stayed flat — `settings: dict`. The caller (`TestClient`) doesn't pass it; FastAPI does.
3. `/config` returned the dict from the dep. Same machinery would work for a database connection, a logged-in user, a feature flag check, etc.

For a config object that doesn't change between requests, building it on every call is wasteful. We'll fix that with `@lru_cache` in notebook 4.3. For now, simplicity over optimization.

## 3. Dependencies That Depend on Other Dependencies

Dependencies are themselves regular callables, so they can `Depends(...)` on others. FastAPI walks the chain and runs everything in the right order — caching repeated deps per-request (covered in §5).

A practical example: a `get_current_portfolio` dep that needs `get_settings` to know which currency to filter on. The route only asks for the portfolio; FastAPI quietly resolves settings first.

In [ ]:
# Pretend store of portfolios keyed by id.
PORTFOLIOS = {
    1: {"id": 1, "name": "Growth", "currency": "USD"},
    2: {"id": 2, "name": "Income", "currency": "EUR"},
}

def get_portfolio(portfolio_id: int, settings: dict = Depends(get_settings)) -> dict:
    p = PORTFOLIOS.get(portfolio_id)
    if p is None:
        return {"error": "not found", "looked_for": portfolio_id}
    # Demonstrate that the chained dep is available here.
    return {**p, "default_page_size": settings["default_page_size"]}

@app.get("/portfolios/{portfolio_id}")
def read_portfolio(portfolio: dict = Depends(get_portfolio)):
    # The route never says it needs settings — but get_portfolio does, so FastAPI resolves the chain.
    return portfolio

r = client.get("/portfolios/1")
print(r.status_code, r.json())
r2 = client.get("/portfolios/99")
print(r2.status_code, r2.json())

Two subtleties worth noticing:

- `get_portfolio` declares `portfolio_id: int` as a regular parameter. FastAPI pulls it from the **path**, exactly as if the route had declared it. Deps can consume request data, not just other deps.
- The route signature only mentions `portfolio`. The settings chain is fully invisible at the route level — which is the whole point. The route asks for the thing it cares about and trusts the dep graph to assemble it.

## 4. Class-Callable Dependencies

A dep can be **any callable**, including a class. When you pass a class to `Depends(...)`, FastAPI calls the constructor with the resolved parameters and passes the instance to the route.

This is the cleanest way to bundle a set of related parameters that several routes share — pagination, sorting, filters.

In [ ]:
from typing import Annotated
from fastapi import Query

class Pagination:
    def __init__(
        self,
        page: int = Query(1, ge=1),
        size: int = Query(20, ge=1, le=100),
    ):
        self.page = page
        self.size = size
        self.offset = (page - 1) * size

@app.get("/holdings")
def list_holdings(pg: Pagination = Depends()):  # Depends() with no arg uses the type annotation
    return {"page": pg.page, "size": pg.size, "offset": pg.offset}

r = client.get("/holdings?page=3&size=25")
print(r.status_code, r.json())
r_default = client.get("/holdings")
print(r_default.json())
r_invalid = client.get("/holdings?page=0")
print(r_invalid.status_code, r_invalid.json()["detail"][0]["msg"])

Three properties of class-callable deps that matter in practice:

- **`Depends()` with no argument** uses the parameter's type annotation as the dep. Saves typing `Depends(Pagination)` while reading the same.
- **Query / Path / Header validation lives in `__init__`**. The OpenAPI schema picks it up automatically — the `/holdings` endpoint now documents `page` and `size` with their constraints.
- **The instance is fresh per request.** No state leaks across requests, but within a single request the instance is reused if multiple deps reference it (next section).

In modern FastAPI you'll also see `Annotated[Pagination, Depends()]` instead of `Pagination = Depends()`. Both work; the `Annotated` form composes better with other type-annotated helpers and is the documented direction. We'll use the simpler form in this notebook for readability.

## 5. Per-Request Caching

If two parts of the dep graph ask for the same dep, FastAPI calls it **once per request** and reuses the result. This is on by default. It matters for performance (one DB connection per request, not three) and for correctness (multiple parts of the handler seeing the same view of state).

You can opt out per-Depends with `Depends(get_x, use_cache=False)` if you really want a fresh value each time, but this is rare.

In [ ]:
call_count = {"settings": 0}

def get_settings_counted() -> dict:
    call_count["settings"] += 1
    return {"currency": "USD", "default_page_size": 20, "call": call_count["settings"]}

def dep_a(s: dict = Depends(get_settings_counted)) -> str:
    return f"a sees call {s['call']}"

def dep_b(s: dict = Depends(get_settings_counted)) -> str:
    return f"b sees call {s['call']}"

@app.get("/cache-demo")
def cache_demo(a: str = Depends(dep_a), b: str = Depends(dep_b)):
    return {"a": a, "b": b, "settings_calls_total": call_count["settings"]}

call_count["settings"] = 0
r = client.get("/cache-demo")
print(r.json())
# Both deps saw the *same* settings dict; the counter only ticked once.

r2 = client.get("/cache-demo")
print(r2.json())
# A NEW request -> the cache is reset, counter ticks again.

The output shows the contract:

- Within request 1, `dep_a` and `dep_b` both consume `get_settings_counted`. It runs **once**; both see the same returned dict.
- Across two requests, the dep runs twice — once per request. The cache is request-scoped, not process-scoped.

This is exactly what you want for things like "the current user", "the open DB transaction", "the request id" — everything in the handler should see the same view of those for the lifetime of the request.

## 6. Yielding Dependencies (cleanup)

For dependencies that own a resource — a DB connection, an open file, a span in a tracer — you need a way to **clean up** after the request. The pattern is a `yield` inside the dep:

```python
def get_db():
    conn = connect()
    try:
        yield conn         # request body runs with conn
    finally:
        conn.close()       # always runs, even on exceptions
```

FastAPI treats this as a generator-style dep: it runs until the `yield`, passes the yielded value to the route, then resumes after the response is sent (or after an exception) to run the teardown.

The same shape works for `async def` deps using `yield` — they integrate cleanly with `httpx.AsyncClient` deps, async DB drivers, etc.

In [ ]:
# A toy connection class with explicit open/close so we can SEE the lifecycle.
class FakeConn:
    def __init__(self):
        self.open = True
        self.log: list[str] = []
    def query(self, sql: str) -> list[str]:
        if not self.open:
            raise RuntimeError("query on a closed connection")
        self.log.append(sql)
        return [f"row_for({sql})"]
    def close(self):
        self.open = False

OPEN_CONNS: list[FakeConn] = []  # so we can inspect lifecycle from outside the request

def get_db():
    conn = FakeConn()
    OPEN_CONNS.append(conn)
    try:
        yield conn
    finally:
        conn.close()

@app.get("/holdings/{portfolio_id}")
def list_holdings_for(portfolio_id: int, db: FakeConn = Depends(get_db)):
    rows = db.query(f"SELECT * FROM holdings WHERE portfolio_id={portfolio_id}")
    return {"rows": rows, "conn_open_during_handler": db.open}

OPEN_CONNS.clear()
r = client.get("/holdings/1")
print("response:", r.json())
print("conn state after request: open =", OPEN_CONNS[-1].open, "queries =", OPEN_CONNS[-1].log)

Two things to verify in the output:

- **`conn_open_during_handler: true`** — inside the route, the connection is live and queryable.
- **`open = False` after the response** — `finally` ran. The teardown is reliable; an exception in the route still goes through `finally`.

A real-world version of this dep would `yield` a connection from a pool, and `finally: pool.release(conn)` instead of `close()`. The mechanics are identical.

## Key Takeaways

- **`Depends(callable)`** is how a route asks for something prepared in advance. The callable can be a function or a class.
- **Deps can depend on other deps.** FastAPI resolves the chain top-down before invoking the route.
- **Class-callable deps** are the clean way to bundle related parameters (pagination, filters). Use `Depends()` (no argument) to lean on the type annotation.
- **Per-request caching is on by default.** Same dep referenced in three places → called once. Across requests → called per request.
- **`yield` deps** give you `try / finally` cleanup. The teardown runs after the response or after an exception. This is the right shape for connections, files, transactions.
- **Capstone tie-in**: the portfolio API will inject a settings object, a current-user object, a repository, and a DB session — all via this same pattern. By the end of chapter 4 you have the full toolkit.

## Exercises

**1. RequestContext dep.** Build a `RequestContext` class-callable dep that captures `request_id` (from a `X-Request-ID` header, fallback to a generated UUID4) and a server-side `start_time` (a `time.perf_counter()` snapshot). Inject it into three routes — one trivial, two using deps that themselves consume `RequestContext`. Confirm that within one request all three see the **same** instance (same `request_id`, same `start_time`).

**2. Yielding dep with an exception.** Modify `get_db` to print `"opened"` before `yield` and `"closed"` after. Add a route that intentionally `raise HTTPException(500)`. Make a request and confirm `closed` is still printed — the `finally` ran. Now add a second route that raises a plain `ValueError`; check what happens (default 500 + still closed).

**3. Override in a "test"** — preview of chapter 7. Use `app.dependency_overrides[get_settings] = lambda: {"currency": "EUR", "default_page_size": 5}` and hit `/portfolios/1`. Confirm the override took effect. Then `app.dependency_overrides.clear()` and confirm you're back to USD. This is the swap-without-touching-the-route pattern that makes FastAPI handlers cheap to test.